# What Is Memory?

In [1]:
import getpass

from openai import OpenAI

openai_key = getpass.getpass()
client = OpenAI(api_key=openai_key)

In [2]:
llm_model = 'gpt-4o'
system_content = 'You are a friendly assistant who will help the user with their queries'
user_content = 'What is the water cycle ?'
response = client.chat.completions.create(
        model=llm_model, messages=[
            {
                "role": "system", "content":
                system_content
            }, {"role": "user", "content": user_content}
        ]
)
print(response.choices[0].message.content)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [3]:
complaint_received = "Mera order abhi tak nahi aaya. Mai Kya karun"
desired_tone = "friendly and helpful"
prompt = f"""Respond to the complaint received in English that is delimited by single quotes with a tone that is {desired_tone}. text: '{complaint_received}'"""
print(prompt)

Respond to the complaint received in English that is delimited by single quotes with a tone that is friendly and helpful. text: 'Mera order abhi tak nahi aaya. Mai Kya karun'


In [4]:
response = client.chat.completions.create(
        model=llm_model, messages=[
            {"role": "system", "content": system_content}, {"role": "user", "content": prompt}
        ]
)
print(response.choices[0].message.content)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [5]:
from langchain_openai import ChatOpenAI

chat_call = ChatOpenAI(temperature=0.0, model=llm_model)
chat_call

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [6]:
template_string = """
Respond to the complaint received in English that is delimited by single quotes with a tone that is {tone}. text: '{complaint}'
"""

In [7]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(template_string)
prompt_template.messages[0].prompt

PromptTemplate(input_variables=['complaint', 'tone'], input_types={}, partial_variables={}, template="\nRespond to the complaint received in English that is delimited by single quotes with a tone that is {tone}. text: '{complaint}'\n")

In [8]:
prompt_template.messages[0].prompt.input_variables

['complaint', 'tone']

In [9]:
user_request = prompt_template.format_messages(tone=desired_tone, complaint=complaint_received)
print((user_request))

[HumanMessage(content="\nRespond to the complaint received in English that is delimited by single quotes with a tone that is friendly and helpful. text: 'Mera order abhi tak nahi aaya. Mai Kya karun'\n", additional_kwargs={}, response_metadata={})]


In [10]:
customer_response = chat_call.invoke(user_request)
print(customer_response.content)

NameError: name 'chat_call' is not defined

In [11]:
product_description = """
The amazing Hercules sofa is a classic 3-seater sofa in velvet texture. Easy to clean with modern style, the sofa comes in five colours - blue, yellow, olive, grey and white Uplift the vibe of your living room in just 999 euros with the new Hercules.
"""

In [12]:
description_template = """
For the following text, extract the information in the format described below:

Brand: If the brand name is mentioned, extract its value else the value is 'unknown'
Item: If the furniture item is mentioned, extract its value else the value is 'unknown'
Size: If the dimensions are mentioned then extract those; if the seating capacity is mentioned extract that; otherwise the value is 'unknown'
Texture: If the texture is mentioned, extract its value else the value is 'unknown'
Colours: If any colour is mentioned, extract the colour names else the value is 'unknown'
Price: If the price is mentioned, extract its value along with the currency else the value is 'unknown'

Format the final output as a JSON with the following keys:
'Brand'
'Item'
'Size'
'Texture'
'Colours'
'Price'

text: {text}
"""

In [13]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(description_template)
print(prompt_template)

input_variables=['text'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template="\nFor the following text, extract the information in the format described below:\n\nBrand: If the brand name is mentioned, extract its value else the value is 'unknown'\nItem: If the furniture item is mentioned, extract its value else the value is 'unknown'\nSize: If the dimensions are mentioned then extract those; if the seating capacity is mentioned extract that; otherwise the value is 'unknown'\nTexture: If the texture is mentioned, extract its value else the value is 'unknown'\nColours: If any colour is mentioned, extract the colour names else the value is 'unknown'\nPrice: If the price is mentioned, extract its value along with the currency else the value is 'unknown'\n\nFormat the final output as a JSON with the following keys:\n'Brand'\n'Item'\n'Size'\n'Texture'\n'Colours'\n'Price'\n\ntex

In [14]:
messages = prompt_template.format_messages(text=product_description)
chat = ChatOpenAI(temperature=0.0, model=llm_model)
response = chat(messages)
print(response)

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [15]:
from langchain_classic.output_parsers import (
    ResponseSchema,
    StructuredOutputParser,
    format_instructions
)


In [16]:
brand_schema = ResponseSchema(
        name="Brand", description="If the brand name is mentioned, extract its value else the value is 'unknown'"
)

item_schema = ResponseSchema(
        name="Item", description="If the furniture item is mentioned, extract its value else the value is 'unknown'"
)

size_schema = ResponseSchema(
        name="Size",
        description="If the dimensions are mentioned then extract those; if the seating capacity is mentioned extract that; otherwise the value is 'unknown'"
)

texture_schema = ResponseSchema(
        name="Texture", description="If the texture is mentioned, extract its value else the value is 'unknown'"
)

colours_schema = ResponseSchema(
        name="Colours", description="If any colour is mentioned, extract the colour names else the value is 'unknown'"
)

price_schema = ResponseSchema(
        name="Price",
        description="If the price is mentioned, extract its value along with the currency else the value is 'unknown'"
)

response_schema = [
    brand_schema,
    item_schema,
    size_schema,
    texture_schema,
    colours_schema,
    price_schema
]

In [17]:
prompt_template = ChatPromptTemplate.from_template(
        template=description_template
)

format_message = prompt_template.format_messages(
        text=product_description,
        format_instructions=format_instructions
)

print(format_message[0].content)


For the following text, extract the information in the format described below:

Brand: If the brand name is mentioned, extract its value else the value is 'unknown'
Item: If the furniture item is mentioned, extract its value else the value is 'unknown'
Size: If the dimensions are mentioned then extract those; if the seating capacity is mentioned extract that; otherwise the value is 'unknown'
Texture: If the texture is mentioned, extract its value else the value is 'unknown'
Colours: If any colour is mentioned, extract the colour names else the value is 'unknown'
Price: If the price is mentioned, extract its value along with the currency else the value is 'unknown'

Format the final output as a JSON with the following keys:
'Brand'
'Item'
'Size'
'Texture'
'Colours'
'Price'

text: 
The amazing Hercules sofa is a classic 3-seater sofa in velvet texture. Easy to clean with modern style, the sofa comes in five colours - blue, yellow, olive, grey and white Uplift the vibe of your living roo

In [18]:
op_parser = StructuredOutputParser.from_response_schemas(response_schema)
output = op_parser.parse(response.content)
print(output)

NameError: name 'response' is not defined